# ECLIPSE — Dark Proteome Prioritization Score (DPPS)

This notebook implements the **DPPS** composite scoring system on top of the existing ECLIPSE pipeline output (`eclipse.csv` / `ECLIPSE.csv`).

It picks up directly after Part I and does **not** re-run MMseqs2 or the Atlas mapping.

### Input files required
| File | Description |
|------|-------------|
| `eclipse.csv` | Output of ECLIPSE Part I (with ESKAPE diversity metrics) |
| `AFDB90v4_subgraphs_summary.csv` | Atlas component summary (used in Part I) |
| *(optional)* `mdr_strains.txt` | One accession/strain ID per line — MDR strain list for S₅ |
| *(optional)* `foldseek_results.tsv` | Foldseek/TED output with novelty calls for S₄ |

### DPPS formula
$$\text{DPPS} = w_1 S_1 + w_2 S_2 + w_3 S_3 + w_4 S_4 + w_5 S_5 + B$$

| Sub-score | Description | Weight |
|-----------|-------------|--------|
| $S_1$ | Darkness penalty: $1 - D_c/100$ | 0.20 |
| $S_2$ | ESKAPE enrichment: $p_{\text{ESKAPE}} \times (1 - E_{\text{genus}})$ | 0.25 |
| $S_3$ | Taxonomic specificity: $1 - E_{\text{relative}}$ | 0.20 |
| $S_4$ | Structural novelty (TED/Foldseek binary/continuous) | 0.20 |
| $S_5$ | MDR strain conservation fraction | 0.15 |
| $B$  | Bonus flags: co-localisation with AMR genes, signal peptide, transmembrane topology | ≤0.15 |

**Tiers:** I (DPPS ≥ 0.75) · II (0.50–0.75) · III (0.25–0.50) · IV (< 0.25)

---
## 0. Imports and configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import Counter
import math
import os
import warnings
warnings.filterwarnings('ignore')

# ── File paths ──────────────────────────────────────────────────────────────
ECLIPSE_CSV        = './eclipse.csv'           # Part I output
COMPONENTS_CSV     = './AFDB90v4_subgraphs_summary.csv'
MDR_STRAINS_FILE   = './mdr_strains.txt'       # optional — set to None to skip S5
FOLDSEEK_TSV       = './foldseek_results.tsv'  # optional — set to None to skip S4

# ── DPPS weights (must sum to 1.0) ─────────────────────────────────────────
WEIGHTS = {
    'S1_darkness':       0.20,
    'S2_eskape_enrich':  0.25,
    'S3_specificity':    0.20,
    'S4_struct_novelty': 0.20,
    'S5_mdr_conserv':    0.15,
}
assert abs(sum(WEIGHTS.values()) - 1.0) < 1e-9, 'Weights must sum to 1.0'

# ── Darkness threshold used in Part I ───────────────────────────────────────
DARK_BRIGHTNESS_CUTOFF = 5  # component_brightness <= this is 'dark'

# ── Tier boundaries ─────────────────────────────────────────────────────────
TIER_BINS   = [0, 0.25, 0.50, 0.75, 1.01]
TIER_LABELS = ['IV', 'III', 'II', 'I']
TIER_COLORS = {'I': '#7B2D8B', 'II': '#D85A30', 'III': '#1D9E75', 'IV': '#888780'}

print('Configuration loaded.')
print(f'  Weights: {WEIGHTS}')
print(f'  Dark cutoff: component_brightness <= {DARK_BRIGHTNESS_CUTOFF}%')

---
## 1. Load Part I output

In [ ]:
# Load ECLIPSE Part I results
df = pd.read_csv(ECLIPSE_CSV, index_col=0)

print(f'Loaded {len(df):,} proteins from {ECLIPSE_CSV}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
# Filter to dark components only (consistent with Part I)
df_dark = df.loc[df['component_brightness'] <= DARK_BRIGHTNESS_CUTOFF].copy()
print(f'Dark proteins (component_brightness <= {DARK_BRIGHTNESS_CUTOFF}%): {len(df_dark):,}')
print(f'Unique dark components: {df_dark["componentID"].nunique():,}')

In [ ]:
# Aggregate to component-level for scoring
# Each component gets one DPPS; proteins inherit it from their component
component_cols = ['componentID', 'component_brightness',
                  'ESKAPE_proportion', 'ESKAPE_genus_evenness', 'ESKAPE_relative_evenness']

# Deduplicate to one row per component
comp_df = df_dark[component_cols].drop_duplicates(subset='componentID').copy()
comp_df.reset_index(drop=True, inplace=True)
print(f'Component-level dataframe: {len(comp_df):,} components')
comp_df.head()

---
## 2. Compute sub-scores S1 – S3 (from existing metrics)

In [ ]:
# ── S1: Darkness penalty ────────────────────────────────────────────────────
# Inverted brightness, normalized to [0,1]
# A component with 0% brightness → S1 = 1.0 (maximally dark)
# A component with 5% brightness → S1 = 0.95

comp_df['S1_darkness'] = 1.0 - (comp_df['component_brightness'] / 100.0)
comp_df['S1_darkness'] = comp_df['S1_darkness'].clip(0, 1)

print('S1 — Darkness penalty')
print(comp_df['S1_darkness'].describe().round(3))

In [ ]:
# ── S2: ESKAPE enrichment ───────────────────────────────────────────────────
# Rewards high ESKAPE proportion AND low genus evenness
# (a real pathogen-biased signal, not just broad taxonomic occurrence)
#
# S2 = ESKAPE_proportion × (1 − ESKAPE_genus_evenness)
#
# Max = 1.0 → all proteins from AMR genera, all from ONE genus
# Min = 0.0 → either no ESKAPE proteins, or perfectly spread across all genera

comp_df['S2_eskape_enrich'] = (
    comp_df['ESKAPE_proportion'] * (1.0 - comp_df['ESKAPE_genus_evenness'])
)
comp_df['S2_eskape_enrich'] = comp_df['S2_eskape_enrich'].clip(0, 1)

print('S2 — ESKAPE enrichment')
print(comp_df['S2_eskape_enrich'].describe().round(3))

In [ ]:
# ── S3: Taxonomic specificity ───────────────────────────────────────────────
# Inverted ESKAPE_relative_evenness
# Low relative evenness = component is exclusive to AMR genera (not spread across all life)
# → high S3 (more interesting target)
#
# S3 = 1 − ESKAPE_relative_evenness

comp_df['S3_specificity'] = 1.0 - comp_df['ESKAPE_relative_evenness']
comp_df['S3_specificity'] = comp_df['S3_specificity'].clip(0, 1)

print('S3 — Taxonomic specificity')
print(comp_df['S3_specificity'].describe().round(3))

---
## 3. Compute S4 — Structural novelty (TED / Foldseek)

**Input format expected (`foldseek_results.tsv`):**

| queryID | ted_domain | foldseek_evalue | is_novel |
|---------|-----------|-----------------|----------|
| PA_001  | TED_12345  | 1e-5 | 0 |
| PA_002  | —          | —    | 1 |

`is_novel = 1` → no ECOD/SCOP assignment found (completely new fold) → **S4 = 1.0**  
`is_novel = 0` with e-value available → **S4 = −log10(e-value) / 50**, capped at 0.8 (known fold, penalized proportionally to confidence)  
Missing → **S4 = 0.5** (no structural data, agnostic)


In [ ]:
def compute_S4_from_foldseek(df_proteins, foldseek_path):
    """
    Map per-protein structural novelty scores to component level.
    Returns a Series indexed by componentID.
    """
    fs = pd.read_csv(foldseek_path, sep='\t')
    
    # Compute per-protein S4
    def protein_S4(row):
        if row.get('is_novel', 0) == 1:
            return 1.0
        evalue = row.get('foldseek_evalue', np.nan)
        if pd.notna(evalue) and evalue > 0:
            # More confident structural hit → lower S4 (known fold)
            raw = -math.log10(evalue) / 50.0
            return min(raw, 0.8)  # cap: even confident known folds get some credit
        return 0.5  # no data
    
    fs['S4_protein'] = fs.apply(protein_S4, axis=1)
    
    # Merge with protein→component mapping
    merged = df_proteins[['queryID', 'componentID']].merge(fs[['queryID', 'S4_protein']],
                                                            on='queryID', how='left')
    merged['S4_protein'].fillna(0.5, inplace=True)
    
    # Aggregate to component: take the maximum S4 within each component
    # (a component is interesting if ANY member has a novel fold)
    comp_S4 = merged.groupby('componentID')['S4_protein'].max()
    return comp_S4


if FOLDSEEK_TSV and os.path.exists(FOLDSEEK_TSV):
    comp_S4 = compute_S4_from_foldseek(df_dark, FOLDSEEK_TSV)
    comp_df['S4_struct_novelty'] = comp_df['componentID'].map(comp_S4).fillna(0.5)
    print(f'S4 computed from {FOLDSEEK_TSV}')
else:
    # No structural data available — assign neutral score 0.5
    # Replace with 1.0 for Pseudomonas-specific novel fold representatives
    # once TED analysis is complete.
    comp_df['S4_struct_novelty'] = 0.5
    print('S4: Foldseek file not found — using neutral placeholder (0.5).')
    print('    → Replace with actual TED/Foldseek output to activate S4.')

comp_df['S4_struct_novelty'] = comp_df['S4_struct_novelty'].clip(0, 1)
print(comp_df['S4_struct_novelty'].describe().round(3))

---
## 4. Compute S5 — MDR strain conservation

**Input format (`mdr_strains.txt`):** One strain accession per line (matching the strain IDs in your PATRIC-derived FASTA headers).

S5 = fraction of MDR strains in which the component has at least one representative protein.

In [ ]:
def compute_S5_mdr_conservation(df_proteins, mdr_path):
    """
    For each component, compute the fraction of MDR strains that carry
    at least one protein from that component.

    Assumes queryID format: STRAIN_ACCESSION|PROTEIN_ID
    (standard PATRIC FASTA header structure).
    Adjust the strain extraction logic below if your headers differ.
    """
    with open(mdr_path) as fh:
        mdr_strains = set(line.strip() for line in fh if line.strip())
    
    n_mdr = len(mdr_strains)
    print(f'  MDR strain list: {n_mdr} strains')
    
    # Extract strain accession from queryID
    # Adjust the split logic to match your actual FASTA header format
    df_prot = df_proteins[['queryID', 'componentID']].copy()
    df_prot['strain'] = df_prot['queryID'].str.split('|').str[0]  # ← adjust if needed
    df_prot['is_mdr'] = df_prot['strain'].isin(mdr_strains)
    
    # Per component: how many distinct MDR strains contribute a protein?
    mdr_counts = (
        df_prot[df_prot['is_mdr']]
        .groupby('componentID')['strain']
        .nunique()
    )
    
    S5 = mdr_counts / n_mdr
    return S5.clip(0, 1)


if MDR_STRAINS_FILE and os.path.exists(MDR_STRAINS_FILE):
    comp_S5 = compute_S5_mdr_conservation(df_dark, MDR_STRAINS_FILE)
    comp_df['S5_mdr_conserv'] = comp_df['componentID'].map(comp_S5).fillna(0.0)
    print(f'S5 computed from {MDR_STRAINS_FILE}')
else:
    # Fallback: approximate S5 using alignment identity as a proxy
    # (higher mean fident within a component → more conserved)
    # Replace with actual MDR strain fractions when available.
    fident_by_comp = df_dark.groupby('componentID')['fident'].mean()
    comp_df['S5_mdr_conserv'] = comp_df['componentID'].map(fident_by_comp).fillna(0.0)
    comp_df['S5_mdr_conserv'] = (comp_df['S5_mdr_conserv'] - comp_df['S5_mdr_conserv'].min()) / \
                                  (comp_df['S5_mdr_conserv'].max() - comp_df['S5_mdr_conserv'].min() + 1e-9)
    print('S5: MDR strain file not found — using normalised mean alignment identity as proxy.')
    print('    → Replace with MDR strain fraction for full biological accuracy.')

comp_df['S5_mdr_conserv'] = comp_df['S5_mdr_conserv'].clip(0, 1)
print(comp_df['S5_mdr_conserv'].describe().round(3))

---
## 5. Bonus flags (B)

Each flag adds a fixed bonus. Total B is capped at **0.15**.

| Flag | Bonus | Detection |
|------|-------|----------|
| Co-localised with AMR gene (GCsnap) | +0.07 | `gcsnap_amr_neighbour` column |
| Signal peptide predicted (SignalP/DeepSig) | +0.05 | `signal_peptide` column |
| Transmembrane topology (TMHMM/DeepTMHMM) | +0.05 | `transmembrane` column |

Add these columns to your DataFrame (e.g. from GCsnap output or SignalP results) to activate the bonuses. Absent columns are silently ignored.

In [ ]:
BONUS_FLAGS = {
    'gcsnap_amr_neighbour': 0.07,   # co-localised with annotated AMR gene
    'signal_peptide':       0.05,   # N-terminal signal peptide predicted
    'transmembrane':        0.05,   # transmembrane helix/barrel predicted
}
BONUS_CAP = 0.15

# Aggregate boolean/binary flag columns from protein → component level
# (component gets the flag if ANY member protein carries it)
bonus_total = pd.Series(0.0, index=comp_df.index)

for flag, bonus_value in BONUS_FLAGS.items():
    if flag in df_dark.columns:
        flag_by_comp = df_dark.groupby('componentID')[flag].max()  # 1 if any protein has flag
        comp_flag = comp_df['componentID'].map(flag_by_comp).fillna(0).astype(float)
        bonus_total += comp_flag * bonus_value
        n_flagged = int((comp_flag > 0).sum())
        print(f'  {flag}: {n_flagged:,} components flagged (+{bonus_value})')
    else:
        print(f'  {flag}: column not found — skipped (add column to activate)')

comp_df['B_bonus'] = bonus_total.clip(0, BONUS_CAP)
print(f'\nBonus B: max={comp_df["B_bonus"].max():.3f}, mean={comp_df["B_bonus"].mean():.3f}')

---
## 6. Compute composite DPPS and assign tiers

In [ ]:
# ── Weighted sum ────────────────────────────────────────────────────────────
comp_df['DPPS_raw'] = sum(
    WEIGHTS[col] * comp_df[col] for col in WEIGHTS
)

# Add bonus (already capped)
comp_df['DPPS'] = (comp_df['DPPS_raw'] + comp_df['B_bonus']).clip(0, 1)

# ── Tier assignment ─────────────────────────────────────────────────────────
comp_df['tier'] = pd.cut(
    comp_df['DPPS'],
    bins=TIER_BINS,
    labels=TIER_LABELS,
    right=False
).astype(str)

# ── Sort by DPPS descending ──────────────────────────────────────────────────
comp_df.sort_values('DPPS', ascending=False, inplace=True)
comp_df.reset_index(drop=True, inplace=True)

print('DPPS distribution:')
print(comp_df['DPPS'].describe().round(3))
print('\nTier counts:')
print(comp_df['tier'].value_counts().sort_index())

In [ ]:
# Preview top-ranked components
display_cols = ['componentID', 'DPPS', 'tier',
                'S1_darkness', 'S2_eskape_enrich', 'S3_specificity',
                'S4_struct_novelty', 'S5_mdr_conserv', 'B_bonus',
                'component_brightness', 'ESKAPE_proportion']

comp_df[display_cols].head(20).round(3)

---
## 7. Propagate DPPS back to protein level and save

In [ ]:
# Map component-level scores back to each protein
score_cols = ['componentID', 'DPPS', 'DPPS_raw', 'tier',
              'S1_darkness', 'S2_eskape_enrich', 'S3_specificity',
              'S4_struct_novelty', 'S5_mdr_conserv', 'B_bonus']

df_scored = df_dark.merge(comp_df[score_cols], on='componentID', how='left')
df_scored.sort_values('DPPS', ascending=False, inplace=True)
df_scored.reset_index(drop=True, inplace=True)

print(f'Scored protein-level dataframe: {len(df_scored):,} rows')

# Save full results
df_scored.to_csv('./eclipse_dpps.csv', index=False)
print('Saved: eclipse_dpps.csv')

# Save Tier I candidates separately
tier1 = df_scored.loc[df_scored['tier'] == 'I']
tier1.to_csv('./eclipse_dpps_tier1.csv', index=False)
print(f'Saved: eclipse_dpps_tier1.csv  ({len(tier1):,} proteins in {tier1["componentID"].nunique():,} components)')

# Save component-level table
comp_df.to_csv('./eclipse_dpps_components.csv', index=False)
print('Saved: eclipse_dpps_components.csv')

---
## 8. Visualisations
### 8.1 DPPS distribution by tier

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ── Left: DPPS histogram coloured by tier ───────────────────────────────────
ax = axes[0]
bins = np.linspace(0, 1, 41)
for tier, color in TIER_COLORS.items():
    subset = comp_df.loc[comp_df['tier'] == tier, 'DPPS']
    ax.hist(subset, bins=bins, color=color, alpha=0.85, label=f'Tier {tier} (n={len(subset)})')

for boundary in TIER_BINS[1:-1]:
    ax.axvline(boundary, color='black', linewidth=0.8, linestyle='--', alpha=0.6)

ax.set_xlabel('DPPS', fontsize=11)
ax.set_ylabel('Number of components', fontsize=11)
ax.set_title('DPPS distribution across tiers', fontsize=12)
ax.legend(fontsize=9)
ax.set_facecolor('#F5F5F5')

# ── Right: Tier pie chart ────────────────────────────────────────────────────
ax2 = axes[1]
tier_counts = comp_df['tier'].value_counts().sort_index()
colors_pie = [TIER_COLORS[t] for t in tier_counts.index]
wedges, texts, autotexts = ax2.pie(
    tier_counts,
    labels=[f'Tier {t}\n(n={v})' for t, v in tier_counts.items()],
    colors=colors_pie,
    autopct='%1.1f%%',
    startangle=90,
    pctdistance=0.75
)
for at in autotexts:
    at.set_fontsize(9)
ax2.set_title('Component tier composition', fontsize=12)

fig.tight_layout()
plt.savefig('./dpps_distribution.pdf', bbox_inches='tight', dpi=300)
plt.savefig('./dpps_distribution.png', bbox_inches='tight', dpi=300)
plt.show()
print('Saved: dpps_distribution.pdf / .png')

### 8.2 Scatter: ESKAPE enrichment vs taxonomic specificity, coloured by DPPS

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

scatter = ax.scatter(
    comp_df['S2_eskape_enrich'],
    comp_df['S3_specificity'],
    c=comp_df['DPPS'],
    cmap='plasma',
    vmin=0, vmax=1,
    s=10, alpha=0.7, linewidths=0
)

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('DPPS', fontsize=10)

# Mark tier boundaries as reference lines
ax.axhline(0.5, color='gray', linewidth=0.6, linestyle=':', alpha=0.7)
ax.axvline(0.5, color='gray', linewidth=0.6, linestyle=':', alpha=0.7)

ax.set_xlabel('S2 — ESKAPE enrichment', fontsize=11)
ax.set_ylabel('S3 — Taxonomic specificity', fontsize=11)
ax.set_title('Dark components: enrichment vs. specificity\n(colour = DPPS)', fontsize=12)
ax.set_facecolor('#F8F8F8')
fig.tight_layout()
plt.savefig('./dpps_scatter_s2_s3.pdf', bbox_inches='tight', dpi=300)
plt.savefig('./dpps_scatter_s2_s3.png', bbox_inches='tight', dpi=300)
plt.show()
print('Saved: dpps_scatter_s2_s3.pdf / .png')

### 8.3 Sub-score heatmap — top 30 Tier I components

In [ ]:
top_n = 30
score_display = ['S1_darkness', 'S2_eskape_enrich', 'S3_specificity',
                 'S4_struct_novelty', 'S5_mdr_conserv', 'B_bonus', 'DPPS']

top_comps = comp_df.loc[comp_df['tier'] == 'I'].head(top_n)

if len(top_comps) == 0:
    print('No Tier I components found. Showing top 30 overall.')
    top_comps = comp_df.head(top_n)

heatmap_data = top_comps.set_index('componentID')[score_display]

fig, ax = plt.subplots(figsize=(9, max(4, len(top_comps) * 0.28)))

sns.heatmap(
    heatmap_data,
    ax=ax,
    cmap='YlOrRd',
    vmin=0, vmax=1,
    annot=True, fmt='.2f',
    annot_kws={'size': 7},
    linewidths=0.3,
    linecolor='white',
    cbar_kws={'label': 'Score [0–1]', 'shrink': 0.6}
)

ax.set_xlabel('')
ax.set_ylabel('Component ID', fontsize=10)
ax.set_title(f'Sub-score heatmap — top {len(top_comps)} Tier I components', fontsize=12)
ax.set_xticklabels(['S1 dark', 'S2 ESKAPE', 'S3 spec.', 'S4 struct', 'S5 MDR', 'Bonus', 'DPPS'],
                   rotation=30, ha='right', fontsize=9)
ax.tick_params(axis='y', labelsize=7)

fig.tight_layout()
plt.savefig('./dpps_heatmap_tier1.pdf', bbox_inches='tight', dpi=300)
plt.savefig('./dpps_heatmap_tier1.png', bbox_inches='tight', dpi=300)
plt.show()
print('Saved: dpps_heatmap_tier1.pdf / .png')

### 8.4 Radar / spider plot — profile of a single top candidate

In [ ]:
def radar_plot(component_id, comp_df, ax=None):
    """
    Draw a radar chart showing the five sub-score profile for one component.
    """
    labels = ['S1\nDarkness', 'S2\nESKAPE', 'S3\nSpecificity',
              'S4\nStructure', 'S5\nMDR']
    score_keys = ['S1_darkness', 'S2_eskape_enrich', 'S3_specificity',
                  'S4_struct_novelty', 'S5_mdr_conserv']
    
    row = comp_df.loc[comp_df['componentID'] == component_id]
    if len(row) == 0:
        print(f'Component {component_id} not found.')
        return
    
    values = [float(row[k].values[0]) for k in score_keys]
    values += values[:1]  # close the loop
    
    N = len(labels)
    angles = [n / float(N) * 2 * math.pi for n in range(N)] + [0]
    
    if ax is None:
        fig, ax = plt.subplots(figsize=(4, 4), subplot_kw=dict(polar=True))
    
    ax.plot(angles, values, color='#57257F', linewidth=2)
    ax.fill(angles, values, color='#57257F', alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(['0.25', '0.5', '0.75', '1.0'], fontsize=7, color='gray')
    dpps_val = float(row['DPPS'].values[0])
    tier_val = str(row['tier'].values[0])
    ax.set_title(f'Component {component_id}\nDPPS={dpps_val:.3f}  Tier {tier_val}',
                 fontsize=10, pad=15)
    return ax


# Plot radar for the top 4 components
top4_ids = comp_df.head(4)['componentID'].tolist()
fig, axes = plt.subplots(1, len(top4_ids), figsize=(4 * len(top4_ids), 4),
                         subplot_kw=dict(polar=True))

if len(top4_ids) == 1:
    axes = [axes]

for ax, comp_id in zip(axes, top4_ids):
    radar_plot(comp_id, comp_df, ax=ax)

fig.suptitle('Sub-score profiles — top 4 components', fontsize=12, y=1.02)
fig.tight_layout()
plt.savefig('./dpps_radar_top4.pdf', bbox_inches='tight', dpi=300)
plt.savefig('./dpps_radar_top4.png', bbox_inches='tight', dpi=300)
plt.show()
print('Saved: dpps_radar_top4.pdf / .png')

### 8.5 Comparison with original hard-filter approach

In [ ]:
# Original Part I filter thresholds — adjust to match what was used in Part I
ORIGINAL_ESKAPE_PROP_CUTOFF = 0.5   # 50% ESKAPE proportion threshold
ORIGINAL_GENUS_EVEN_CUTOFF  = 0.5   # below this = Pseudomonas-specific

# Original binary selection
original_selected = comp_df[
    (comp_df['ESKAPE_proportion'] >= ORIGINAL_ESKAPE_PROP_CUTOFF)
]

# DPPS Tier I
dpps_tier1 = comp_df[comp_df['tier'] == 'I']

# Overlap analysis
orig_set  = set(original_selected['componentID'])
dpps_set  = set(dpps_tier1['componentID'])
overlap   = orig_set & dpps_set
dpps_only = dpps_set - orig_set
orig_only = orig_set - dpps_set

print('=== Coverage comparison ===')
print(f'Original hard filter:  {len(orig_set):>5,} components')
print(f'DPPS Tier I:           {len(dpps_set):>5,} components')
print(f'Shared:                {len(overlap):>5,} components')
print(f'New in DPPS Tier I:    {len(dpps_only):>5,} components (rescued by S4/S5/bonus)')
print(f'Dropped by DPPS:       {len(orig_only):>5,} components (low S3/S4/S5)')

fig, ax = plt.subplots(figsize=(5, 4))
from matplotlib_venn import venn2
try:
    venn2([orig_set, dpps_set],
          set_labels=('Original filter', 'DPPS Tier I'),
          set_colors=('#888780', '#7B2D8B'), alpha=0.6, ax=ax)
    ax.set_title('Overlap: original filter vs. DPPS Tier I', fontsize=12)
    fig.tight_layout()
    plt.savefig('./dpps_venn.pdf', bbox_inches='tight', dpi=300)
    plt.savefig('./dpps_venn.png', bbox_inches='tight', dpi=300)
    plt.show()
    print('Saved: dpps_venn.pdf / .png')
except ImportError:
    print('matplotlib_venn not installed — run: pip install matplotlib-venn')
    print('Skipping Venn diagram. Summary stats above are still valid.')

---
## 9. Summary report

In [ ]:
print('=' * 60)
print('ECLIPSE DPPS — Summary')
print('=' * 60)
print(f'\nInput:  {len(df):,} proteins from eclipse.csv')
print(f'Dark (brightness <= {DARK_BRIGHTNESS_CUTOFF}%): {len(df_dark):,} proteins, {df_dark["componentID"].nunique():,} components')
print()

for tier in ['I', 'II', 'III', 'IV']:
    n_comp  = int((comp_df['tier'] == tier).sum())
    n_prot  = int((df_scored['tier'] == tier).sum())
    mean_dp = comp_df.loc[comp_df['tier'] == tier, 'DPPS'].mean()
    print(f'  Tier {tier}: {n_comp:>5,} components  |  {n_prot:>7,} proteins  |  mean DPPS = {mean_dp:.3f}')

print()
print('Top 10 prioritised components:')
print(comp_df[['componentID', 'DPPS', 'tier',
               'S1_darkness', 'S2_eskape_enrich', 'S3_specificity',
               'S4_struct_novelty', 'S5_mdr_conserv']].head(10).round(3).to_string(index=False))

print('\nOutput files:')
for f in ['eclipse_dpps.csv', 'eclipse_dpps_tier1.csv', 'eclipse_dpps_components.csv',
          'dpps_distribution.pdf', 'dpps_scatter_s2_s3.pdf',
          'dpps_heatmap_tier1.pdf', 'dpps_radar_top4.pdf']:
    exists = '✓' if os.path.exists(f'./{f}') else '○'
    print(f'  {exists}  {f}')

---
## 10. Weight sensitivity analysis (optional)

Assess how robust the top-ranked components are to perturbations in DPPS weights.

In [ ]:
# Monte Carlo weight perturbation — resample weights 500 times
# and record how often each component lands in Tier I

N_BOOT = 500
np.random.seed(42)

score_matrix = comp_df[list(WEIGHTS.keys())].values  # shape (n_components, 5)

tier1_freq = np.zeros(len(comp_df))

for _ in range(N_BOOT):
    # Dirichlet sample — random weights that sum to 1
    w = np.random.dirichlet(np.ones(5))
    dpps_boot = score_matrix @ w + comp_df['B_bonus'].values
    dpps_boot = np.clip(dpps_boot, 0, 1)
    tier1_freq += (dpps_boot >= 0.75).astype(float)

comp_df['tier1_stability'] = tier1_freq / N_BOOT

print(f'Tier I stability (fraction of {N_BOOT} random weight sets) — top 20:')
stable = comp_df.nlargest(20, 'DPPS')[['componentID', 'DPPS', 'tier', 'tier1_stability']]
print(stable.round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(comp_df['DPPS'], comp_df['tier1_stability'],
           c=[TIER_COLORS.get(t, 'gray') for t in comp_df['tier']],
           s=8, alpha=0.6)
ax.axhline(0.8, linestyle='--', color='gray', linewidth=0.8, label='80% stability')
ax.axvline(0.75, linestyle='--', color='gray', linewidth=0.8)
ax.set_xlabel('DPPS (nominal weights)', fontsize=11)
ax.set_ylabel('Tier I stability (fraction of weight draws)', fontsize=11)
ax.set_title('DPPS weight sensitivity analysis', fontsize=12)
patches = [mpatches.Patch(color=c, label=f'Tier {t}') for t, c in TIER_COLORS.items()]
ax.legend(handles=patches, fontsize=8)
ax.set_facecolor('#F8F8F8')
fig.tight_layout()
plt.savefig('./dpps_sensitivity.pdf', bbox_inches='tight', dpi=300)
plt.savefig('./dpps_sensitivity.png', bbox_inches='tight', dpi=300)
plt.show()
print('Saved: dpps_sensitivity.pdf / .png')